# 03 — M2 High/Low classifier

Logistic vs LDA, repeated stratified CV, and the leakage guard.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BACKEND = Path.cwd()
if not (BACKEND / "src").exists():
    BACKEND = BACKEND.parent
sys.path.insert(0, str(BACKEND / "src"))
PROCESSED = BACKEND / "data" / "processed"
REPORTS = BACKEND / "reports"


In [2]:
from evision.models import m2_classify
features = pd.read_csv(PROCESSED / "user_city_features.csv")
m2 = m2_classify.fit(features, repeats=5)
print("n =", m2.n, "| positives =", m2.positives, "| label =", m2.label_col)
display(m2.comparison.round(3))

n = 42 | positives = 21 | label = effective_public_demand


,model,auc_mean,auc_sd,auc_p05,auc_p95,train_auc,train_cv_gap,accuracy,f1,precision,recall,loocv_accuracy
0,LDA,0.838,0.133,0.565,1.0,0.916,0.078,0.768,0.737,0.820,0.704,0.762
1,Logistic,0.828,0.133,0.570,1.0,0.918,0.091,0.792,0.777,0.825,0.754,0.786


## Leakage check

The label is built from demand and chargers, so none of these may be predictors.

In [3]:
forbidden = {"ev_stock", "stations_10km", "fast_10km", "growth_12m",
             "effective_public_demand", "effective_public_demand_10km"}
print("overlap:", forbidden & set(m2_classify.EXOGENOUS))
print("predictors:", m2_classify.EXOGENOUS)

overlap: set()
predictors: ['log_density', 'log_income', 'fuel_price_inr_per_litre', 'ev_policy_score', 'highway_corridor', 'summer_max_temp_c']


## Sensitivity: policy and unverified demographics

In [4]:
display(pd.read_csv(PROCESSED / "model_m2_sensitivity.csv").round(3))

,model,auc_mean,auc_sd,auc_p05,auc_p95,train_auc,train_cv_gap,accuracy,f1,precision,recall,loocv_accuracy,sample
0,Logistic,0.822,0.136,0.6,1.0,0.883,0.061,0.765,0.740,0.806,0.714,0.765,all_51
1,LDA,0.818,0.138,0.6,1.0,0.877,0.059,0.768,0.745,0.807,0.720,0.784,all_51
2,LDA,0.837,0.127,0.6,1.0,0.878,0.042,0.779,0.755,0.817,0.730,0.804,without_policy
3,Logistic,0.832,0.135,0.6,1.0,0.885,0.052,0.771,0.748,0.807,0.728,0.784,without_policy
4,Logistic,0.896,0.109,0.7,1.0,0.934,0.038,0.813,0.805,0.835,0.805,0.818,harmonised_demographics
5,LDA,0.892,0.118,0.7,1.0,0.926,0.034,0.802,0.795,0.823,0.799,0.818,harmonised_demographics
